# 02. SNUH Foundation Tokenization ETL

이 노트북은 확정된 Phase 1 과업 5~9를 한 흐름으로 실행합니다.

5. LAB token 규칙 적용
6. train-only token registry 생성
7. PostgreSQL 내부 집계와 토큰화
8. FERMAT 4-column shard 생성
9. split overlap, token coverage, sequence length 검증

기존 감사 노트북의 임시 분석 셀을 이어 붙이지 않습니다. 모든 설정과 산출물은 이 파일과 output manifest에 기록됩니다.

기본값은 전체 환자의 1%를 선택하는 end-to-end pilot입니다. 성공 후 `PATIENT_BUCKETS = 100`으로 바꾸면 동일한 코드로 전체 실행합니다.

In [ ]:
# Pod 이미지에 없을 때만 한 번 실행하세요. 외부 인터넷이 차단된 경우
# 플랫폼 내부 패키지 저장소에서 설치되지 않을 수 있습니다.
import importlib.util

required_packages = ["psycopg", "numpy", "pandas", "pyarrow", "psutil"]
missing_packages = [
    package for package in required_packages
    if importlib.util.find_spec(package) is None
]
if missing_packages:
    raise RuntimeError(
        "Missing Pod packages: " + ", ".join(missing_packages)
        + ". Install them from the platform package source before continuing."
    )
print("Required packages: OK")


In [ ]:
import getpass
import hashlib
import json
import os
import platform
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import psycopg
from psycopg import sql

# ------------------------------------------------------------------
# Reproducible configuration
# ------------------------------------------------------------------
DB_HOST = "pg-2vge6u.vpc-cdb-kr.gov-ntruss.com"
DB_PORT = 5432
DB_NAME = "cdm"
DB_USER = "jaegyun_jung"
DB_SCHEMA = "cdm2024_official"
DB_SSLMODE = "disable"
DB_END_DATE = "2025-02-05"

SPLIT_SEED = 42
PILOT_SEED = 20260611
TRAIN_MAX_BUCKET = 70
VAL_MAX_BUCKET = 85

# 1 = 1% patient pilot. Set to 100 only after pilot verification.
PATIENT_BUCKETS = 1
assert 1 <= PATIENT_BUCKETS <= 100

# Negative values are clinically valid for some measurements (for example,
# base excess and central venous pressure). Drop negatives only for audited
# concepts whose physical scale cannot be negative; these were observed as
# local sentinel values such as -1 in the pilot audit.
NONNEGATIVE_MEASUREMENT_IDS = [
    3001376,   # Heart rate by pulse oximetry
    3004249,   # Systolic blood pressure
    3012888,   # Diastolic blood pressure
    3020891,   # Body temperature
    3024171,   # Respiratory rate
    3025315,   # Body weight
    3027018,   # Heart rate
    3036277,   # Body height
    40762499,  # Oxygen saturation
    21492241,  # Mean noninvasive blood pressure
    21490852,  # Invasive mean blood pressure
    44789316,  # Ambulatory diastolic blood pressure
]

# Numeric LAB becomes value-binned only if both thresholds are met.
# Otherwise a LAB_TEST token preserves test occurrence.
MIN_NUMERIC_ROWS_FOR_BINS = 1000
MIN_NUMERIC_PATIENTS_FOR_BINS = 100
N_LAB_BINS = 10

# Domain token frequency thresholds, computed on train patients only.
MIN_DX_ROWS = 30
MIN_RX_ROWS = 30
MIN_PX_ROWS = 30
MIN_LAB_CAT_ROWS = 30

# Same patient/date/test/unit numeric duplicates are collapsed to median.
COLLAPSE_NUMERIC_DAILY = "median"

OUTPUT_ROOT = Path("outputs/snuh_tokenization_etl")
RUN_NAME = f"patient_{PATIENT_BUCKETS:03d}pct_seed_{SPLIT_SEED}"
OUTPUT_DIR = OUTPUT_ROOT / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DB_PASSWORD = os.environ.get("SNUH_CDM_PASSWORD")
if not DB_PASSWORD:
    DB_PASSWORD = getpass.getpass("SNUH CDM password: ")

print("Run:", RUN_NAME)
print("Python:", platform.python_version())
print("CPU logical cores:", psutil.cpu_count(logical=True))
print("RAM GB:", round(psutil.virtual_memory().total / 1024**3, 1))
print("Disk free GB:", round(shutil.disk_usage(OUTPUT_DIR).free / 1024**3, 1))
print("Output:", OUTPUT_DIR.resolve())


## A. Connect and create session-local cohort

서버 영구 schema에는 아무것도 만들지 않습니다. 한 DB session 안의 temporary table만 사용합니다.

In [ ]:
conn = psycopg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    sslmode=DB_SSLMODE,
    connect_timeout=15,
    application_name="fermat_tokenization_etl",
    options="-c statement_timeout=0 -c temp_buffers=256MB",
)
conn.autocommit = True

def query_df(query, params=None):
    started = time.time()
    with conn.cursor() as cur:
        cur.execute(query, params or ())
        columns = [item.name for item in cur.description]
        rows = cur.fetchall()
    frame = pd.DataFrame(rows, columns=columns)
    print(f"Completed in {time.time() - started:,.1f}s; rows={len(frame):,}")
    return frame

def execute(query, params=None, label="SQL"):
    started = time.time()
    with conn.cursor() as cur:
        cur.execute(query, params or ())
    print(f"{label}: {time.time() - started:,.1f}s")

identity = query_df("""
SELECT current_user, current_database(), pg_backend_pid(),
       has_schema_privilege(%s, 'USAGE') AS schema_usage
""", (DB_SCHEMA,))
display(identity)


In [ ]:
execute("DROP TABLE IF EXISTS tmp_fermat_person")
execute(sql.SQL("""
CREATE TEMP TABLE tmp_fermat_person ON COMMIT PRESERVE ROWS AS
WITH assigned AS (
    SELECT
        p.person_id,
        p.gender_concept_id,
        p.year_of_birth,
        p.month_of_birth,
        p.day_of_birth,
        mod(
            hashtextextended(p.person_id::text, %s)
                & 9223372036854775807,
            100
        )::integer AS split_bucket,
        mod(
            hashtextextended(p.person_id::text, %s)
                & 9223372036854775807,
            100
        )::integer AS pilot_bucket
    FROM {}.person AS p
    WHERE p.year_of_birth BETWEEN 1900 AND 2025
)
SELECT
    person_id,
    gender_concept_id,
    year_of_birth,
    month_of_birth,
    day_of_birth,
    split_bucket,
    pilot_bucket,
    CASE
        WHEN split_bucket < %s THEN 'train'
        WHEN split_bucket < %s THEN 'val'
        ELSE 'test'
    END AS split
FROM assigned
WHERE pilot_bucket < %s
""").format(sql.Identifier(DB_SCHEMA)),
    (SPLIT_SEED, PILOT_SEED, TRAIN_MAX_BUCKET, VAL_MAX_BUCKET, PATIENT_BUCKETS),
    "Create patient cohort",
)
execute("CREATE INDEX ON tmp_fermat_person(person_id)", label="Index cohort")

split_counts = query_df("""
SELECT split, COUNT(*) AS patients
FROM tmp_fermat_person
GROUP BY split
ORDER BY split
""")
display(split_counts)
split_counts.to_csv(OUTPUT_DIR / "split_counts.csv", index=False)

assert set(split_counts.split) == {'train', 'val', 'test'}, split_counts


## B. Train-only domain frequency and LAB rules

이 단계에서만 vocabulary 포함 여부와 LAB cutpoint를 결정합니다. Validation/test 정보는 사용하지 않습니다.

In [ ]:
# Standard-concept frequencies for DX/RX/PX.
DOMAIN_SPECS = {
    "DX": ("condition_occurrence", "condition_concept_id", "condition_start_date", MIN_DX_ROWS),
    "RX": ("drug_exposure", "drug_concept_id", "drug_exposure_start_date", MIN_RX_ROWS),
    "PX": ("procedure_occurrence", "procedure_concept_id", "procedure_date", MIN_PX_ROWS),
}

domain_frequency_frames = []
for token_type, (table, concept_col, date_col, min_rows) in DOMAIN_SPECS.items():
    frame = query_df(sql.SQL("""
    SELECT
        %s::text AS token_type,
        e.{}::bigint AS concept_id,
        COUNT(*)::bigint AS rows,
        COUNT(DISTINCT e.person_id)::bigint AS patients
    FROM {}.{} AS e
    JOIN tmp_fermat_person AS p USING (person_id)
    WHERE p.split = 'train'
      AND e.{} BETWEEN DATE '1900-01-01' AND %s::date
      AND e.{} IS NOT NULL
      AND e.{} <> 0
    GROUP BY e.{}
    HAVING COUNT(*) >= %s
    ORDER BY rows DESC, concept_id
    """).format(
        sql.Identifier(concept_col),
        sql.Identifier(DB_SCHEMA),
        sql.Identifier(table),
        sql.Identifier(date_col),
        sql.Identifier(concept_col),
        sql.Identifier(concept_col),
        sql.Identifier(concept_col),
    ), (token_type, DB_END_DATE, min_rows))
    frame.to_parquet(OUTPUT_DIR / f"train_{token_type.lower()}_frequency.parquet", index=False)
    domain_frequency_frames.append(frame)

domain_frequencies = pd.concat(domain_frequency_frames, ignore_index=True)
display(domain_frequencies.groupby('token_type').agg(tokens=('concept_id', 'size'), rows=('rows', 'sum')))


In [ ]:
# Numeric LAB is first collapsed to one median per patient/date/test/unit.
execute("DROP TABLE IF EXISTS tmp_numeric_lab_daily")
execute(sql.SQL("""
CREATE TEMP TABLE tmp_numeric_lab_daily ON COMMIT PRESERVE ROWS AS
SELECT
    m.person_id,
    m.measurement_date AS event_date,
    m.measurement_concept_id,
    COALESCE(m.unit_concept_id, 0) AS unit_concept_id,
    percentile_cont(0.5) WITHIN GROUP (ORDER BY m.value_as_number)
        AS value_as_number
FROM {}.measurement AS m
JOIN tmp_fermat_person AS p USING (person_id)
WHERE m.value_as_number IS NOT NULL
  AND m.measurement_concept_id IS NOT NULL
  AND m.measurement_concept_id <> 0
  AND m.measurement_date BETWEEN DATE '1900-01-01' AND %s::date
  AND NOT (
      m.measurement_concept_id = ANY(%s)
      AND m.value_as_number < 0
  )
GROUP BY
    m.person_id,
    m.measurement_date,
    m.measurement_concept_id,
    COALESCE(m.unit_concept_id, 0)
""").format(sql.Identifier(DB_SCHEMA)),
    (DB_END_DATE, NONNEGATIVE_MEASUREMENT_IDS),
    "Collapse numeric LAB",
)
execute("CREATE INDEX ON tmp_numeric_lab_daily(person_id)", label="Index numeric LAB")

numeric_lab_stats = query_df("""
SELECT
    n.measurement_concept_id,
    n.unit_concept_id,
    COUNT(*)::bigint AS daily_rows,
    COUNT(DISTINCT n.person_id)::bigint AS patients
FROM tmp_numeric_lab_daily AS n
JOIN tmp_fermat_person AS p USING (person_id)
WHERE p.split = 'train'
GROUP BY n.measurement_concept_id, n.unit_concept_id
ORDER BY daily_rows DESC
""")
numeric_lab_stats["is_frequent"] = (
    (numeric_lab_stats.daily_rows >= MIN_NUMERIC_ROWS_FOR_BINS)
    & (numeric_lab_stats.patients >= MIN_NUMERIC_PATIENTS_FOR_BINS)
)
numeric_lab_stats.to_parquet(OUTPUT_DIR / "train_numeric_lab_stats.parquet", index=False)
display(numeric_lab_stats.head(30))
print("Frequent numeric test-unit pairs:", int(numeric_lab_stats.is_frequent.sum()))


In [ ]:
# Compute train-only decile cutpoints only for frequent pairs.
frequent_pairs = numeric_lab_stats.loc[
    numeric_lab_stats.is_frequent,
    ["measurement_concept_id", "unit_concept_id"],
].copy()

execute("DROP TABLE IF EXISTS tmp_frequent_lab_pair")
execute("""
CREATE TEMP TABLE tmp_frequent_lab_pair (
    measurement_concept_id bigint,
    unit_concept_id bigint
) ON COMMIT PRESERVE ROWS
""")
with conn.cursor() as cur:
    cur.executemany(
        "INSERT INTO tmp_frequent_lab_pair VALUES (%s, %s)",
        list(frequent_pairs.itertuples(index=False, name=None)),
    )

lab_cutpoints = query_df("""
SELECT
    n.measurement_concept_id,
    n.unit_concept_id,
    percentile_cont(ARRAY[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])
        WITHIN GROUP (ORDER BY n.value_as_number) AS cutpoints
FROM tmp_numeric_lab_daily AS n
JOIN tmp_fermat_person AS p USING (person_id)
JOIN tmp_frequent_lab_pair AS f
  ON f.measurement_concept_id = n.measurement_concept_id
 AND f.unit_concept_id = n.unit_concept_id
WHERE p.split = 'train'
GROUP BY n.measurement_concept_id, n.unit_concept_id
ORDER BY n.measurement_concept_id, n.unit_concept_id
""")
lab_cutpoints.to_parquet(OUTPUT_DIR / "train_lab_decile_cutpoints.parquet", index=False)
display(lab_cutpoints.head())


In [ ]:
# Categorical LAB combinations observed in train.
lab_categorical_frequency = query_df(sql.SQL("""
SELECT
    m.measurement_concept_id::bigint,
    m.value_as_concept_id::bigint,
    COUNT(*)::bigint AS rows,
    COUNT(DISTINCT m.person_id)::bigint AS patients
FROM {}.measurement AS m
JOIN tmp_fermat_person AS p USING (person_id)
WHERE p.split = 'train'
  AND m.value_as_concept_id IS NOT NULL
  AND m.value_as_concept_id <> 0
  AND m.measurement_concept_id IS NOT NULL
  AND m.measurement_concept_id <> 0
  AND m.measurement_date BETWEEN DATE '1900-01-01' AND %s::date
GROUP BY m.measurement_concept_id, m.value_as_concept_id
HAVING COUNT(*) >= %s
ORDER BY rows DESC
""").format(sql.Identifier(DB_SCHEMA)), (DB_END_DATE, MIN_LAB_CAT_ROWS))
lab_categorical_frequency.to_parquet(
    OUTPUT_DIR / "train_lab_categorical_frequency.parquet",
    index=False,
)
display(lab_categorical_frequency.head(20))


## C. Build the train-only dense token registry

OMOP concept ID는 임상적 의미 식별자이고, `token_id`는 embedding row를 위한 연속 정수입니다.

In [ ]:
registry_rows = [
    {"token_key": "SPECIAL:PAD", "token_type": "PAD", "token_type_id": 0},
    {"token_key": "SPECIAL:NO_EVENT", "token_type": "NO_EVENT", "token_type_id": 8},
    {"token_key": "SPECIAL:UNK_DX", "token_type": "DX", "token_type_id": 1},
    {"token_key": "SPECIAL:UNK_RX", "token_type": "RX", "token_type_id": 2},
    {"token_key": "SPECIAL:UNK_PX", "token_type": "PX", "token_type_id": 3},
    {"token_key": "SPECIAL:UNK_LAB", "token_type": "LAB", "token_type_id": 4},
    {"token_key": "SPECIAL:DEATH", "token_type": "DTH", "token_type_id": 6},
    {"token_key": "SEX:M", "token_type": "SEX", "token_type_id": 7},
    {"token_key": "SEX:F", "token_type": "SEX", "token_type_id": 7},
    {"token_key": "SEX:U", "token_type": "SEX", "token_type_id": 7},
]

type_id = {"DX": 1, "RX": 2, "PX": 3, "LAB": 4}
for row in domain_frequencies.itertuples(index=False):
    registry_rows.append({
        "token_key": f"{row.token_type}:{int(row.concept_id)}",
        "token_type": row.token_type,
        "token_type_id": type_id[row.token_type],
    })

for row in numeric_lab_stats.itertuples(index=False):
    base = f"LAB:{int(row.measurement_concept_id)}:{int(row.unit_concept_id)}"
    if row.is_frequent:
        for bin_number in range(1, N_LAB_BINS + 1):
            registry_rows.append({
                "token_key": f"{base}:Q{bin_number:02d}",
                "token_type": "LAB",
                "token_type_id": 4,
            })
    else:
        registry_rows.append({
            "token_key": f"LAB_TEST:{int(row.measurement_concept_id)}",
            "token_type": "LAB",
            "token_type_id": 4,
        })

for row in lab_categorical_frequency.itertuples(index=False):
    registry_rows.append({
        "token_key": (
            f"LAB_CAT:{int(row.measurement_concept_id)}:"
            f"{int(row.value_as_concept_id)}"
        ),
        "token_type": "LAB",
        "token_type_id": 4,
    })

token_registry = (
    pd.DataFrame(registry_rows)
    .drop_duplicates("token_key")
    .reset_index(drop=True)
)
token_registry.insert(0, "token_id", np.arange(len(token_registry), dtype=np.uint32))
token_registry.to_parquet(OUTPUT_DIR / "token_registry.parquet", index=False)
token_registry.to_csv(OUTPUT_DIR / "token_registry.csv", index=False)
display(token_registry.groupby("token_type").size().rename("tokens"))


## D. Materialize compact event tables in the DB session

Pilot에서는 temporary table을 생성해 검증합니다. 전체 실행에서는 이 구간의 runtime과 temp-storage 사용량을 먼저 확인해야 합니다.

In [ ]:
# Upload the dense registry and cutpoints to session-local temp tables.
execute("DROP TABLE IF EXISTS tmp_token_registry")
execute("""
CREATE TEMP TABLE tmp_token_registry (
    token_id bigint,
    token_key text,
    token_type text,
    token_type_id integer
) ON COMMIT PRESERVE ROWS
""")
with conn.cursor() as cur:
    cur.executemany(
        "INSERT INTO tmp_token_registry VALUES (%s,%s,%s,%s)",
        list(token_registry[["token_id", "token_key", "token_type", "token_type_id"]]
             .itertuples(index=False, name=None)),
    )
execute("CREATE INDEX ON tmp_token_registry(token_key)", label="Index registry")

execute("DROP TABLE IF EXISTS tmp_lab_cutpoint")
execute("""
CREATE TEMP TABLE tmp_lab_cutpoint (
    measurement_concept_id bigint,
    unit_concept_id bigint,
    cutpoints double precision[]
) ON COMMIT PRESERVE ROWS
""")
with conn.cursor() as cur:
    cur.executemany(
        "INSERT INTO tmp_lab_cutpoint VALUES (%s,%s,%s)",
        list(lab_cutpoints.itertuples(index=False, name=None)),
    )


In [ ]:
# Create compact unified events. age_in_days uses a deterministic birth-date
# fallback: month=7 and day=1 when unavailable.
execute("DROP TABLE IF EXISTS tmp_fermat_event")
execute(sql.SQL("""
CREATE TEMP TABLE tmp_fermat_event ON COMMIT PRESERVE ROWS AS
WITH birth AS (
    SELECT
        person_id,
        make_date(
            year_of_birth,
            CASE WHEN month_of_birth BETWEEN 1 AND 12 THEN month_of_birth ELSE 7 END,
            CASE WHEN day_of_birth BETWEEN 1 AND 28 THEN day_of_birth ELSE 1 END
        ) AS birth_date,
        gender_concept_id,
        split
    FROM tmp_fermat_person
),
domain_event AS (
    SELECT e.person_id, e.condition_start_date AS event_date,
           'DX:' || e.condition_concept_id::text AS token_key, 1 AS type_order
    FROM {}.condition_occurrence e JOIN birth b USING(person_id)
    WHERE e.condition_start_date BETWEEN b.birth_date AND %s::date
      AND e.condition_concept_id <> 0
    UNION ALL
    SELECT e.person_id, e.drug_exposure_start_date,
           'RX:' || e.drug_concept_id::text, 2
    FROM {}.drug_exposure e JOIN birth b USING(person_id)
    WHERE e.drug_exposure_start_date BETWEEN b.birth_date AND %s::date
      AND e.drug_concept_id <> 0
    UNION ALL
    SELECT e.person_id, e.procedure_date,
           'PX:' || e.procedure_concept_id::text, 3
    FROM {}.procedure_occurrence e JOIN birth b USING(person_id)
    WHERE e.procedure_date BETWEEN b.birth_date AND %s::date
      AND e.procedure_concept_id <> 0
),
numeric_lab AS (
    SELECT
        n.person_id,
        n.event_date,
        CASE
            WHEN c.cutpoints IS NULL THEN
                'LAB_TEST:' || n.measurement_concept_id::text
            ELSE
                'LAB:' || n.measurement_concept_id::text || ':' ||
                n.unit_concept_id::text || ':Q' ||
                lpad((1 + (
                    SELECT COUNT(*) FROM unnest(c.cutpoints) AS q(v)
                    WHERE n.value_as_number > q.v
                ))::text, 2, '0')
        END AS token_key,
        4 AS type_order
    FROM tmp_numeric_lab_daily n
    LEFT JOIN tmp_lab_cutpoint c
      ON c.measurement_concept_id = n.measurement_concept_id
     AND c.unit_concept_id = n.unit_concept_id
),
categorical_lab AS (
    SELECT DISTINCT
        m.person_id,
        m.measurement_date AS event_date,
        'LAB_CAT:' || m.measurement_concept_id::text || ':' ||
            m.value_as_concept_id::text AS token_key,
        4 AS type_order
    FROM {}.measurement m
    JOIN birth b USING(person_id)
    WHERE m.measurement_date BETWEEN b.birth_date AND %s::date
      AND m.measurement_concept_id <> 0
      AND m.value_as_concept_id IS NOT NULL
      AND m.value_as_concept_id <> 0
),
death_event AS (
    SELECT d.person_id, d.death_date AS event_date,
           'SPECIAL:DEATH' AS token_key, 6 AS type_order
    FROM {}.death d JOIN birth b USING(person_id)
    WHERE d.death_date BETWEEN b.birth_date AND %s::date
),
sex_event AS (
    SELECT person_id, birth_date AS event_date,
           CASE gender_concept_id
               WHEN 8507 THEN 'SEX:M'
               WHEN 8532 THEN 'SEX:F'
               ELSE 'SEX:U'
           END AS token_key,
           0 AS type_order
    FROM birth
),
all_event AS (
    SELECT * FROM sex_event
    UNION ALL SELECT * FROM domain_event
    UNION ALL SELECT * FROM numeric_lab
    UNION ALL SELECT * FROM categorical_lab
    UNION ALL SELECT * FROM death_event
)
SELECT
    a.person_id,
    (a.event_date - b.birth_date)::integer AS age_in_days,
    COALESCE(r.token_id,
        CASE
            WHEN a.token_key LIKE 'DX:%%' THEN 2
            WHEN a.token_key LIKE 'RX:%%' THEN 3
            WHEN a.token_key LIKE 'PX:%%' THEN 4
            ELSE 5
        END
    )::bigint AS token_id,
    COALESCE(r.token_type_id, a.type_order)::integer AS token_type_id,
    b.split,
    a.event_date,
    a.type_order
FROM all_event a
JOIN birth b USING(person_id)
LEFT JOIN tmp_token_registry r USING(token_key)
WHERE a.event_date IS NOT NULL
  AND a.event_date >= b.birth_date
""").format(
    sql.Identifier(DB_SCHEMA),
    sql.Identifier(DB_SCHEMA),
    sql.Identifier(DB_SCHEMA),
    sql.Identifier(DB_SCHEMA),
    sql.Identifier(DB_SCHEMA),
), (DB_END_DATE, DB_END_DATE, DB_END_DATE, DB_END_DATE, DB_END_DATE),
"Materialize compact events")
execute("CREATE INDEX ON tmp_fermat_event(split, person_id, event_date)", label="Index events")

event_summary = query_df("""
SELECT
    split,
    token_type_id,
    COUNT(*) AS events,
    COUNT(DISTINCT person_id) AS patients,
    COUNT(*) FILTER (WHERE token_id IN (2,3,4,5)) AS unknown_events
FROM tmp_fermat_event
GROUP BY split, token_type_id
ORDER BY split, token_type_id
""")
display(event_summary)
event_summary.to_csv(OUTPUT_DIR / "event_summary.csv", index=False)


## E. Stream FERMAT 4-column binary shards

각 split은 `(patient_id_dense, age_in_days, token_id, token_type_id)` uint32 binary로 저장됩니다. 원래 OMOP person_id는 `patient_id_map.parquet`에 별도로 보존합니다.

In [ ]:
execute("DROP TABLE IF EXISTS tmp_patient_map")
execute("""
CREATE TEMP TABLE tmp_patient_map ON COMMIT PRESERVE ROWS AS
SELECT
    row_number() OVER (ORDER BY person_id) - 1 AS patient_id_dense,
    person_id,
    split
FROM tmp_fermat_person
""", label="Create patient map")
execute("CREATE INDEX ON tmp_patient_map(person_id)", label="Index patient map")

patient_map = query_df("""
SELECT patient_id_dense, person_id, split
FROM tmp_patient_map
ORDER BY patient_id_dense
""")
patient_map.to_parquet(OUTPUT_DIR / "patient_id_map.parquet", index=False)

# Named server-side cursors require a transaction. Temporary tables use
# ON COMMIT PRESERVE ROWS, so committing after each streamed shard is safe.
conn.autocommit = False

def stream_split(split_name, chunk_size=250_000):
    output_path = OUTPUT_DIR / f"{split_name}.bin"
    row_count = 0
    with conn.cursor(name=f"stream_{split_name}") as cur, open(output_path, "wb") as handle:
        cur.execute("""
        SELECT
            p.patient_id_dense,
            e.age_in_days,
            e.token_id,
            e.token_type_id
        FROM tmp_fermat_event e
        JOIN tmp_patient_map p USING(person_id)
        WHERE e.split = %s
        ORDER BY e.person_id, e.event_date, e.type_order, e.token_id
        """, (split_name,))
        while True:
            rows = cur.fetchmany(chunk_size)
            if not rows:
                break
            array = np.asarray(rows, dtype=np.uint32)
            array.tofile(handle)
            row_count += len(array)
            if row_count % 1_000_000 < chunk_size:
                print(split_name, f"{row_count:,} events")
    conn.commit()
    return {"split": split_name, "events": row_count, "path": str(output_path)}

available_splits = split_counts.split.tolist()
shard_results = [stream_split(split_name) for split_name in available_splits]
display(pd.DataFrame(shard_results))


## F. Validate artifacts

In [ ]:
validation_rows = []
for result in shard_results:
    path = Path(result["path"])
    raw = np.memmap(path, dtype=np.uint32, mode="r")
    assert raw.size % 4 == 0
    data = raw.reshape(-1, 4)
    patient_ids = data[:, 0]
    lengths = np.diff(
        np.r_[0, np.flatnonzero(np.diff(patient_ids)) + 1, len(patient_ids)]
    ) if len(data) else np.array([], dtype=int)
    validation_rows.append({
        "split": result["split"],
        "events": len(data),
        "patients_with_events": int(len(np.unique(patient_ids))) if len(data) else 0,
        "min_age_days": int(data[:, 1].min()) if len(data) else None,
        "max_age_days": int(data[:, 1].max()) if len(data) else None,
        "max_token_id": int(data[:, 2].max()) if len(data) else None,
        "median_sequence_length": float(np.median(lengths)) if len(lengths) else 0,
        "p95_sequence_length": float(np.percentile(lengths, 95)) if len(lengths) else 0,
        "max_sequence_length": int(lengths.max()) if len(lengths) else 0,
        "file_gb": path.stat().st_size / 1024**3,
    })

validation = pd.DataFrame(validation_rows)
display(validation)
validation.to_csv(OUTPUT_DIR / "shard_validation.csv", index=False)

if set(available_splits) == {"train", "val", "test"}:
    split_sets = {
        split: set(patient_map.loc[patient_map.split == split, "person_id"])
        for split in available_splits
    }
    assert not (split_sets["train"] & split_sets["val"])
    assert not (split_sets["train"] & split_sets["test"])
    assert not (split_sets["val"] & split_sets["test"])
    print("Patient overlap: 0")

total_event_rows = event_summary.events.sum()
total_unknown_rows = event_summary.unknown_events.sum()
print("Unknown event rate:", total_unknown_rows / total_event_rows)


In [ ]:
config = {
    "db_host": DB_HOST,
    "db_schema": DB_SCHEMA,
    "db_end_date": DB_END_DATE,
    "split_seed": SPLIT_SEED,
    "patient_buckets": PATIENT_BUCKETS,
    "pilot_seed": PILOT_SEED,
    "split_rule": "split hash mod 100 gives 70/15/15; independent pilot hash selects patient percent",
    "min_numeric_rows_for_bins": MIN_NUMERIC_ROWS_FOR_BINS,
    "min_numeric_patients_for_bins": MIN_NUMERIC_PATIENTS_FOR_BINS,
    "n_lab_bins": N_LAB_BINS,
    "collapse_numeric_daily": COLLAPSE_NUMERIC_DAILY,
    "min_dx_rows": MIN_DX_ROWS,
    "min_rx_rows": MIN_RX_ROWS,
    "min_px_rows": MIN_PX_ROWS,
    "min_lab_cat_rows": MIN_LAB_CAT_ROWS,
    "stored_token_id_min": int(token_registry.token_id.min()),
    "stored_token_id_max": int(token_registry.token_id.max()),
    "model_vocab_size": int(token_registry.token_id.max()) + 2,
    "token_id_note": "FERMAT get_batch adds 1; model id 0 is padding",
    "nonnegative_measurement_ids": NONNEGATIVE_MEASUREMENT_IDS,
    "python_version": platform.python_version(),
    "cpu_count": psutil.cpu_count(logical=True),
    "ram_gb": round(psutil.virtual_memory().total / 1024**3, 2),
}
with open(OUTPUT_DIR / "manifest.json", "w", encoding="utf-8") as handle:
    json.dump(config, handle, indent=2)

checksums = {}
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        digest = hashlib.sha256()
        with open(path, "rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        checksums[path.name] = digest.hexdigest()
with open(OUTPUT_DIR / "sha256.json", "w", encoding="utf-8") as handle:
    json.dump(checksums, handle, indent=2)

print(json.dumps(config, indent=2))
print("Artifacts:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name, f"{path.stat().st_size / 1024**2:.2f} MB")


In [ ]:
conn.close()
DB_PASSWORD = None
print("Connection closed. Phase 1 tokenization ETL run complete.")
